# FREM-S12 Kaggle campaign launcher
This notebook delegates to the hash-bound package. Qualification and generated smoke do not consume market outcomes. Scientific cells remain receipt-gated.

In [ ]:
from pathlib import Path

MODE = "qualify-kaggle"  # qualify-kaggle | generated-smoke | run-search-cell | select-search | run-cell | analyze
EXECUTE = False
INPUT = Path("/kaggle/input/frem-s12-input")
OUTPUT = Path("/kaggle/working/frem-s12-output")
FAMILY = "size_bm"
FOLD = "A"
TRIAL = 0
SEED = 11
TOTAL_SHARDS = 8
SHARD = 0

In [ ]:
import hashlib
import json
import platform
import subprocess
import sys

OUTPUT.mkdir(parents=True, exist_ok=True)
input_files = {
    path.name: {"sha256": hashlib.sha256(path.read_bytes()).hexdigest(), "size": path.stat().st_size}
    for path in sorted(INPUT.glob("*"))
    if path.is_file()
}
preflight = {
    "schema_version": "frem-s12-kaggle-preflight-v1",
    "scientific_admitted": False,
    "python_version": platform.python_version(),
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "input_exists": INPUT.is_dir(),
    "input_files": input_files,
}
(OUTPUT / "preflight.json").write_text(json.dumps(preflight, indent=2, sort_keys=True) + "\n")
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"Unsupported Kaggle Python {platform.python_version()}; required 3.12; preflight receipt preserved")

wheels = sorted(INPUT.glob("*.whl"))
if not wheels:
    raise RuntimeError("No attached wheels; fail closed")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--no-deps",
        "--find-links",
        str(INPUT),
        "frem-s12==0.1.0",
    ],
    check=True,
)

In [ ]:
import os

OUTPUT.mkdir(parents=True, exist_ok=True)
base = [sys.executable, "-m", "frem_s12.cli"]
if MODE == "qualify-kaggle":
    command = [*base, "qualify-kaggle", "--output", str(OUTPUT / "environment.json")]
elif MODE == "generated-smoke":
    command = [*base, "generated-smoke", "--seed", str(SEED), "--output", str(OUTPUT / "generated-smoke")]
elif MODE == "run-search-cell":
    if not EXECUTE:
        raise RuntimeError("Scientific execution requires explicit EXECUTE=True")
    command = [
        *base,
        "run-search-cell",
        "--execute",
        "--trial",
        str(TRIAL),
        "--family",
        FAMILY,
        "--seed",
        str(SEED),
        "--archive",
        str(INPUT / f"{FAMILY}-201608.zip"),
        "--data-receipt",
        str(INPUT / f"{FAMILY}-data.json"),
        "--environment-receipt",
        str(INPUT / "environment.json"),
        "--consumption-root",
        str(OUTPUT / "consumption"),
        "--output",
        str(OUTPUT / f"trial={TRIAL}" / f"family={FAMILY}" / f"seed={SEED}"),
    ]
elif MODE == "select-search":
    command = [
        *base,
        "select-search",
        "--campaign",
        str(INPUT / "complete-search"),
        "--output",
        str(OUTPUT / "selection.json"),
    ]
elif MODE == "run-cell":
    if not EXECUTE:
        raise RuntimeError("Scientific execution requires explicit EXECUTE=True")
    command = [
        *base,
        "run-cell",
        "--execute",
        "--family",
        FAMILY,
        "--fold",
        FOLD,
        "--seed",
        str(SEED),
        "--archive",
        str(INPUT / f"{FAMILY}-201608.zip"),
        "--data-receipt",
        str(INPUT / f"{FAMILY}-data.json"),
        "--environment-receipt",
        str(INPUT / "environment.json"),
        "--selection-receipt",
        str(INPUT / "selection.json"),
        "--consumption-root",
        str(OUTPUT / "consumption"),
        "--checkpoint-dir",
        str(OUTPUT / "checkpoints"),
        "--output",
        str(OUTPUT / f"family={FAMILY}" / f"fold={FOLD}" / f"seed={SEED}"),
    ]
elif MODE == "analyze":
    command = [
        *base,
        "analyze",
        "--campaign",
        str(INPUT / "complete-campaign"),
        "--output",
        str(OUTPUT / "decision.json"),
    ]
else:
    raise RuntimeError(f"Unknown MODE: {MODE}")
print(" ".join(command))
subprocess.run(command, check=True, env={**os.environ, "PYTHONHASHSEED": "0"})
if MODE == "qualify-kaggle":
    import json

    qualification = json.loads((OUTPUT / "environment.json").read_text())
    if qualification.get("scientific_admitted") is not True:
        raise RuntimeError("Kaggle scientific qualification failed closed; preserve and inspect environment.json")

In [ ]:
import hashlib
import json

files = {
    str(path.relative_to(OUTPUT)): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in sorted(OUTPUT.rglob("*"))
    if path.is_file()
}
print(json.dumps({"mode": MODE, "files": files}, indent=2, sort_keys=True))